## Evaluation of the model

For evaluate the model we will use one of the metrics showed in the paper about The Well Dataset, the **Variance Scaled Mean Square Error** (**VMSE**).

## **VMSE definition**

$$
VMSE = \left< | u - v |^2 \right> / \left( \left< | u - v |^2 \right> + \epsilon  \right),
$$
where $\epsilon = 10^{-7}$

## Step 1: Load the trained model

In [1]:
# !pip install the_well[benchmark]

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from einops import rearrange
from tqdm import tqdm

from the_well.data import WellDataset
from the_well.benchmark.models.unet_convnext import UNetConvNext

device = "cuda"
base_path = "./datasets"  # path/to/storage

/home/franklin/miniconda3/envs/mlearning/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
class CNextbaseline(nn.Module):
    def __init__(self,
                 in_channels: int,
                 out_channels: int,
                 initial_dimension: int,
                 up_down_blocks: int,
                 blocks_per_stage: int,
                 bottleneck_blocks: int,
                 spatial_resolution: tuple,
                 spatial_dims: int=2):
        
        super().__init__()
        
        self.model = UNetConvNext(
            dim_in = in_channels,
            dim_out = out_channels,
            n_spatial_dims = spatial_dims,
            spatial_resolution = spatial_resolution,
            stages = up_down_blocks,
            blocks_per_stage = blocks_per_stage,
            blocks_at_neck = bottleneck_blocks,
            init_features = initial_dimension 
        )
    
    def forward(self, x):
        return self.model(x)

In [3]:
# We already know the number of fields are 11
F = 11

In [4]:
model_cnet = CNextbaseline(in_channels=4*F,
                           out_channels=1*F,
                           initial_dimension=42,
                           blocks_per_stage=2,
                           up_down_blocks=4,
                           bottleneck_blocks=1,
                           spatial_resolution=(256,256)).to(device)

model_cnet_phys = CNextbaseline(in_channels=4*F,
                           out_channels=1*F,
                           initial_dimension=42,
                           blocks_per_stage=2,
                           up_down_blocks=4,
                           bottleneck_blocks=1,
                           spatial_resolution=(256,256)).to(device)

In [5]:
stats = torch.load("./model_UNetConvNext_stats/trained_UNetConvNext_epoch_9.pth")
model_cnet.load_state_dict(stats)

stats_phys = torch.load("./model_UNetConvNext_phys_stats/trained_UNetConvNext_phys_best.pth")
model_cnet_phys.load_state_dict(stats_phys)

<All keys matched successfully>

## Step 2: Evaluate the trained model in the **test split**

In [6]:
# Normalisation stage
norm_params = torch.load('./model_stats/normalisation_train_param.pt')
mu = norm_params['mu']
sigma = norm_params['sigma']

In [7]:
def preprocess(x):
    ''' 
    This function standardise the input data before apply the neural network.
    This function is designed for a tensor of the shape B Ti Lx Ly F
    '''
    return (x - mu) / sigma


def postprocess_field(x, id_field):
    ''' 
    This function return to the state previously the standardisation for a single field.
    This function takes a tensor of the shape B Lx Ly
    '''

    return sigma[id_field] * x + mu[id_field]

In [8]:
print(mu[0], sigma[0])
print(mu, sigma)

tensor(1., device='cuda:0') tensor(0.0010, device='cuda:0')
tensor([ 1.0000e+00,  7.3987e-11,  6.3370e-13,  5.0201e-01, -6.3930e-03,
        -6.3930e-03,  4.9799e-01,  7.9132e-13, -4.1344e-11, -4.1344e-11,
        -7.9132e-13], device='cuda:0') tensor([0.0010, 0.5823, 0.5717, 0.3224, 0.3337, 0.3337, 0.3225, 0.4096, 0.4353,
        0.4353, 0.4096], device='cuda:0')


In [9]:
dataset_test = WellDataset(
    well_base_path = f'{base_path}/datasets',
    well_dataset_name = 'active_matter',
    well_split_name = "test",
    n_steps_input = 4,
    n_steps_output = 1,
    use_normalization = False,
)

In [10]:
# Parameters for the dataset splitting into batches
size = 2
workers = 2

test_loader = torch.utils.data.DataLoader(
    dataset=dataset_test,
    shuffle=True,
    batch_size=size,
    num_workers=workers,
    pin_memory = True,
    persistent_workers=True
)

In [11]:
# Remember the key associated to each field
field_names_tmp = dataset_test.field_names

field_names_tmp2 = [
    name for group in field_names_tmp.values() for name in group
]

field_names = {}

for id in range(len(field_names_tmp2)):
    field_names[id] = field_names_tmp2[id]

print(field_names)

{0: 'concentration', 1: 'velocity_x', 2: 'velocity_y', 3: 'D_xx', 4: 'D_xy', 5: 'D_yx', 6: 'D_yy', 7: 'E_xx', 8: 'E_xy', 9: 'E_yx', 10: 'E_yy'}


In [12]:
# Functions for extracting the fields properly

def concentration_comp(prediction_batch):
    ''' 
    Takes the tensor of the shape B (To F) Lx Ly 
    and returns the postprocessed tensor associated to the concentration component.

    Input:
    - prediction_batch: tensor of B=4 (To=1 F=11) Lx=256 Ly=256 shape

    Output:
    - 1 tensor of B=4 Lx=256 Ly=256 shape, namely rho (concentration) 

    Notes: 
    This function can be applied also to the ground truth because it
    has the same shape as the prediction_batch.

    - This function applies the postprocess function to return to the original.

    '''
    # Choose the concentration field
    c_id = [k for k, v in field_names.items() if v=='concentration']

    c_list = [] 

    for id in range(prediction_batch.shape[0]):
        pred_sample_tmp = prediction_batch[id]

        c_sample = pred_sample_tmp[c_id[0]]

        c_list.append(c_sample)

        c_tensor = torch.stack(c_list, dim = 0) 
        c_postprocess = postprocess_field(c_tensor, c_id[0])
    
    return c_postprocess


def vel_comp(prediction_batch):
    ''' 
    Takes the tensor of the shape B (To F) Lx Ly 
    and returns the tensor associated to the vx and vy components.

    Input:
    - prediction_batch: tensor of B=4 (To=1 F=11) Lx=256 Ly=256 shape

    Output:
    - 2 tensors of B=4 Lx=256 Ly=256 shape, namely vx and vy 

    Note: This function can be applied also to the ground truth because it
    has the same shape as the prediction_batch
    '''
    # Choose the vx and vy components
    vx_id = [k for k, v in field_names.items() if v=='velocity_x']
    vy_id = [k for k, v in field_names.items() if v=='velocity_y']

    vx_list = [] 
    vy_list = []

    for id in range(prediction_batch.shape[0]):
        pred_sample_tmp = prediction_batch[id]

        vx_sample = pred_sample_tmp[vx_id[0]]
        vy_sample = pred_sample_tmp[vy_id[0]]

        vx_list.append(vx_sample)
        vy_list.append(vy_sample)
    
    vx_tensor, vy_tensor =  torch.stack(vx_list, dim = 0), torch.stack(vy_list, dim = 0)
    vx_post, vy_post = postprocess_field(vx_tensor, vx_id[0]), postprocess_field(vy_tensor, vy_id[0])

    return vx_post, vy_post

def orientation_tensor_comp(prediction_batch):
    ''' 
    Takes the tensor of the shape B (To F) Lx Ly 
    and returns the tensor associated to the D_xx, D_xy, D_yx and D_yy components.

    Input:
    - prediction_batch: tensor of B=4 (To=1 F=11) Lx=256 Ly=256 shape

    Output:
    - 4 tensors of B=4 Lx=256 Ly=256 shape, namely Dxx, Dxy, Dyx, and Dyy 
    
    Note: This function can be applied also to the ground truth because it
    has the same shape as the prediction_batch
    '''
    # Choose the components identification
    Dxx_id = [k for k, v in field_names.items() if v=='D_xx']
    Dxy_id = [k for k, v in field_names.items() if v=='D_xy']
    Dyx_id = [k for k, v in field_names.items() if v=='D_yx']
    Dyy_id = [k for k, v in field_names.items() if v=='D_yy']

    Dxx_list = [] 
    Dxy_list = []
    Dyx_list = [] 
    Dyy_list = []
    
    for id in range(prediction_batch.shape[0]):
        pred_sample_tmp = prediction_batch[id]

        Dxx_sample = pred_sample_tmp[Dxx_id[0]]
        Dxy_sample = pred_sample_tmp[Dxy_id[0]]
        Dyx_sample = pred_sample_tmp[Dyx_id[0]]
        Dyy_sample = pred_sample_tmp[Dyy_id[0]]

        Dxx_list.append(Dxx_sample)
        Dxy_list.append(Dxy_sample)
        Dyx_list.append(Dyx_sample)
        Dyy_list.append(Dyy_sample)
    
    Dxx_tensor, Dxy_tensor = torch.stack(Dxx_list, dim = 0), torch.stack(Dxy_list, dim = 0)
    Dyx_tensor, Dyy_tensor = torch.stack(Dyx_list, dim = 0), torch.stack(Dyy_list, dim = 0)

    Dxx_post, Dxy_post = postprocess_field(Dxx_tensor, Dxx_id[0]), postprocess_field(Dxy_tensor, Dxy_id[0])
    Dyx_post, Dyy_post = postprocess_field(Dyx_tensor, Dyx_id[0]), postprocess_field(Dyy_tensor, Dyy_id[0])

    return Dxx_post, Dxy_post, Dyx_post, Dyy_post

def strain_tensor_comp(prediction_batch):
    ''' 
    Takes the tensor of the shape B (To F) Lx Ly 
    and returns the tensor associated to the E_xx, E_xy, E_yx and E_yy components.

    Input:
    - prediction_batch: tensor of B=4 (To=1 F=11) Lx=256 Ly=256 shape

    Output:
    - 4 strain_rate components for the batch: 4 tensors of B=4 Lx=256 Ly=256 shape,
    namely Exx, Exy, Eyx, and Eyy 
    
    Notes:
    - This function is defined for extracting only the strain tensor components 
    from the prediction batch (batch_size = 4)
    - To is the output time (To = 1) as the output is only one

    Remember: a batch consist of group some items from the training set for 
    improve training velocity. Batch -> minigroups at which training set is splitting

    Note: This function can be applied also to the ground truth because it
    has the same shape as the prediction_batch
    '''
    # Choose the components identification
    Exx_id = [k for k, v in field_names.items() if v=='E_xx']
    Exy_id = [k for k, v in field_names.items() if v=='E_xy']
    Eyx_id = [k for k, v in field_names.items() if v=='E_yx']
    Eyy_id = [k for k, v in field_names.items() if v=='E_yy']

    Exx_list = [] 
    Exy_list = []
    Eyx_list = [] 
    Eyy_list = []

    for id in range(prediction_batch.shape[0]):
        pred_sample_tmp = prediction_batch[id]

        Exx_sample = pred_sample_tmp[Exx_id[0]]
        Exy_sample = pred_sample_tmp[Exy_id[0]]
        Eyx_sample = pred_sample_tmp[Eyx_id[0]]
        Eyy_sample = pred_sample_tmp[Eyy_id[0]]

        Exx_list.append(Exx_sample)
        Exy_list.append(Exy_sample)
        Eyx_list.append(Eyx_sample)
        Eyy_list.append(Eyy_sample)

    Exx_tensor, Exy_tensor = torch.stack(Exx_list, dim = 0), torch.stack(Exy_list, dim = 0)
    Eyx_tensor, Eyy_tensor = torch.stack(Eyx_list, dim = 0), torch.stack(Eyy_list, dim = 0) 

    Exx_post, Exy_post = postprocess_field(Exx_tensor, Exx_id[0]), postprocess_field(Exy_tensor, Exy_id[0])
    Eyx_post, Eyy_post = postprocess_field(Eyx_tensor, Eyx_id[0]), postprocess_field(Eyy_tensor, Eyy_id[0])

    return Exx_post, Exy_post, Eyx_post, Eyy_post

$$
VMSE = \left< | u - v |^2 \right> / \left( \left< | u - v |^2 \right> + \epsilon  \right),
$$
where $\epsilon = 10^{-7}$

In [13]:
epsilon = 1e-7

In [14]:
def VMSE_metric(truth, pred):
    numerator = torch.mean(torch.abs(truth - pred)**2)
    denominator = torch.mean(torch.abs(truth)**2) + epsilon

    metric = numerator / denominator

    return metric

In [15]:
VMSE_concentration = [ ]

VMSE_vx = [ ]
VMSE_vy = [ ]

VMSE_Dxx = [ ]
VMSE_Dxy = [ ]
VMSE_Dyx = [ ]
VMSE_Dyy = [ ]

VMSE_Exx = [ ]
VMSE_Exy = [ ]
VMSE_Eyx = [ ]
VMSE_Eyy = [ ]

VMSE_concentration_phys = [ ]

VMSE_vx_phys = [ ]
VMSE_vy_phys = [ ]

VMSE_Dxx_phys = [ ]
VMSE_Dxy_phys = [ ]
VMSE_Dyx_phys = [ ]
VMSE_Dyy_phys = [ ]

VMSE_Exx_phys = [ ]
VMSE_Exy_phys = [ ]
VMSE_Eyx_phys = [ ]
VMSE_Eyy_phys = [ ]

In [16]:
for batch in (bar := tqdm(test_loader, desc=f"Evaluation Epoch {0}", bar_format='{l_bar}{bar}{r_bar}')):
        x = batch["input_fields"]
        x = x.to(device)
        x = preprocess(x)
        x = rearrange(x, "B Ti Lx Ly F -> B (Ti F) Lx Ly")

        y = batch["output_fields"]
        y = y.to(device)
        y = preprocess(y)
        y = rearrange(y, "B To Lx Ly F -> B (To F) Lx Ly")

        fx = model_cnet(x)
        fx_phys = model_cnet_phys(x)

        # Relevant component tensors 
        rho = concentration_comp(fx)
        rho_phys = concentration_comp(fx_phys)
        rho_truth = concentration_comp(y)

        vx, vy = vel_comp(fx)
        vx_phys, vy_phys = vel_comp(fx_phys)
        vx_truth, vy_truth = vel_comp(y)

        D_xx, D_xy, D_yx, D_yy = orientation_tensor_comp(fx)
        D_xx_phys, D_xy_phys, D_yx_phys, D_yy_phys = orientation_tensor_comp(fx_phys)
        D_xx_truth, D_xy_truth, D_yx_truth, D_yy_truth = orientation_tensor_comp(y)

        E_xx, E_xy, E_yx, E_yy = strain_tensor_comp(fx)
        E_xx_phys, E_xy_phys, E_yx_phys, E_yy_phys = strain_tensor_comp(fx_phys) 
        E_xx_truth, E_xy_truth, E_yx_truth, E_yy_truth = strain_tensor_comp(y) 

        # Evaluation stage using VMSE for each field indidually
        rho_vmse = VMSE_metric(rho_truth, rho)
        rho_phys_vmse = VMSE_metric(rho_truth, rho_phys)

        vx_vmse, vy_vmse = VMSE_metric(vx_truth, vx), VMSE_metric(vy_truth, vy)
        vx_phys_vmse, vy_phys_vmse = VMSE_metric(vx_truth, vx_phys), VMSE_metric(vy_truth, vy_phys)

        Dxx_vmse, Dxy_vmse = VMSE_metric(D_xx_truth, D_xx), VMSE_metric(D_xy_truth, D_xy)
        Dyx_vmse, Dyy_vmse = VMSE_metric(D_yx_truth, D_yx), VMSE_metric(D_yy_truth, D_yy)
        Dxx_phys_vmse, Dxy_phys_vmse = VMSE_metric(D_xx_truth, D_xx_phys), VMSE_metric(D_xy_truth, D_xy_phys)
        Dyx_phys_vmse, Dyy_phys_vmse = VMSE_metric(D_yx_truth, D_yx_phys), VMSE_metric(D_yy_truth, D_yy_phys)

        Exx_vmse, Exy_vmse = VMSE_metric(E_xx_truth, E_xx), VMSE_metric(E_xy_truth, E_xy)
        Eyx_vmse, Eyy_vmse = VMSE_metric(E_yx_truth, E_yx), VMSE_metric(E_yy_truth, E_yy)
        Exx_phys_vmse, Exy_phys_vmse = VMSE_metric(E_xx_truth, E_xx_phys), VMSE_metric(E_xy_truth, E_xy_phys)
        Eyx_phys_vmse, Eyy_phys_vmse = VMSE_metric(E_yx_truth, E_yx_phys), VMSE_metric(E_yy_truth, E_yy_phys)

        #Store for analysis
        VMSE_concentration.append(rho_vmse.detach().item())

        VMSE_vx.append(vx_vmse.detach().item())
        VMSE_vy.append(vy_vmse.detach().item())

        VMSE_Dxx.append(Dxx_vmse.detach().item())
        VMSE_Dxy.append(Dxy_vmse.detach().item())
        VMSE_Dyx.append(Dyx_vmse.detach().item())
        VMSE_Dyy.append(Dyy_vmse.detach().item())

        VMSE_Exx.append(Exx_vmse.detach().item())
        VMSE_Exy.append(Exy_vmse.detach().item())
        VMSE_Eyx.append(Eyx_vmse.detach().item())
        VMSE_Eyy.append(Eyy_vmse.detach().item())

        VMSE_concentration_phys.append(rho_phys_vmse.detach().item())

        VMSE_vx_phys.append(vx_phys_vmse.detach().item())
        VMSE_vy_phys.append(vy_phys_vmse.detach().item())

        VMSE_Dxx_phys.append(Dxx_phys_vmse.detach().item())
        VMSE_Dxy_phys.append(Dxy_phys_vmse.detach().item())
        VMSE_Dyx_phys.append(Dyx_phys_vmse.detach().item())
        VMSE_Dyy_phys.append(Dyy_phys_vmse.detach().item())

        VMSE_Exx_phys.append(Exx_phys_vmse.detach().item())
        VMSE_Exy_phys.append(Exy_phys_vmse.detach().item())
        VMSE_Eyx_phys.append(Eyx_phys_vmse.detach().item())
        VMSE_Eyy_phys.append(Eyy_phys_vmse.detach().item())

        vmse_val = np.mean(np.array([rho_vmse.detach().item(), vx_vmse.detach().item(), vy_vmse.detach().item(), Dxx_vmse.detach().item(),\
                                     Dxy_vmse.detach().item(), Dyx_vmse.detach().item(), Dyy_vmse.detach().item(), \
                                        Exx_vmse.detach().item(), Exy_vmse.detach().item(), Eyx_vmse.detach().item(), Eyy_vmse.detach().item()]))
        
        vmse_val_phys = np.mean(np.array([rho_phys_vmse.detach().item(), vx_phys_vmse.detach().item(), vy_phys_vmse.detach().item(), Dxx_phys_vmse.detach().item(),\
                                     Dxy_phys_vmse.detach().item(), Dyx_phys_vmse.detach().item(), Dyy_phys_vmse.detach().item(), \
                                        Exx_phys_vmse.detach().item(), Exy_phys_vmse.detach().item(), Eyx_phys_vmse.detach().item(), Eyy_phys_vmse.detach().item()]))
        
        bar.set_postfix(vmse=vmse_val, vmse_phys=vmse_val_phys)

Evaluation Epoch 0: 100%|██████████| 1001/1001 [02:52<00:00,  5.80it/s, vmse=0.00205, vmse_phys=0.0131] 


In [17]:
torch.cuda.empty_cache()

In [18]:
VMSE_concentration = np.array(VMSE_concentration)

VMSE_vx = np.array(VMSE_vx)
VMSE_vy = np.array(VMSE_vy)

VMSE_Dxx = np.array(VMSE_Dxx)
VMSE_Dxy = np.array(VMSE_Dxy)
VMSE_Dyx = np.array(VMSE_Dyx)
VMSE_Dyy = np.array(VMSE_Dyy)

VMSE_Exx = np.array(VMSE_Exx)
VMSE_Exy = np.array(VMSE_Exy)
VMSE_Eyx = np.array(VMSE_Eyx)
VMSE_Eyy = np.array(VMSE_Eyy)

VMSE_concentration_phys = np.array(VMSE_concentration_phys)

VMSE_vx_phys = np.array(VMSE_vx_phys)
VMSE_vy_phys = np.array(VMSE_vy_phys)

VMSE_Dxx_phys = np.array(VMSE_Dxx_phys)
VMSE_Dxy_phys = np.array(VMSE_Dxy_phys)
VMSE_Dyx_phys = np.array(VMSE_Dyx_phys)
VMSE_Dyy_phys = np.array(VMSE_Dyy_phys)

VMSE_Exx_phys = np.array(VMSE_Exx_phys)
VMSE_Exy_phys = np.array(VMSE_Exy_phys)
VMSE_Eyx_phys = np.array(VMSE_Eyx_phys)
VMSE_Eyy_phys = np.array(VMSE_Eyy_phys)

In [29]:
print('Evaluation metric VMSE for the UNetConvNext and UNetConvNext with physics: \n ')
print(f' Field         |          VMSE           |        VMSE-physics ')
print(f' concentration |  {np.mean(VMSE_concentration)} | {np.mean(VMSE_concentration_phys)}')
print(f'     vx        |  {np.mean(VMSE_vx)}  | {np.mean(VMSE_vx_phys)} ')
print(f'     vy        |  {np.mean(VMSE_vy)}  | {np.mean(VMSE_vy_phys)} ')
print(f'     Dxx       |  {np.mean(VMSE_Dxx)} | {np.mean(VMSE_Dxx_phys)}')
print(f'     Dxy       |  {np.mean(VMSE_Dxy)}  | {np.mean(VMSE_Dxy_phys)} ')
print(f'     Dyx       |  {np.mean(VMSE_Dyx)}  | {np.mean(VMSE_Dyx_phys)} ')
print(f'     Dyy       |  {np.mean(VMSE_Dyy)}  | {np.mean(VMSE_Dyy_phys)} ')
print(f'     Exx       |  {np.mean(VMSE_Exx)}   | {np.mean(VMSE_Exx_phys)} ')
print(f'     Exy       |  {np.mean(VMSE_Exy)}   | {np.mean(VMSE_Exy_phys)} ')
print(f'     Eyx       |  {np.mean(VMSE_Eyx)}  | {np.mean(VMSE_Eyx_phys)} ')
print(f'     Eyy       |  {np.mean(VMSE_Eyy)}   | {np.mean(VMSE_Eyy_phys)} ')

Evaluation metric VMSE for the UNetConvNext and UNetConvNext with physics: 
 
 Field         |          VMSE           |        VMSE-physics 
 concentration |  1.9415010025909823e-09 | 8.145914808391626e-07
     vx        |  0.0059088217332226725  | 0.08140107572402482 
     vy        |  0.0019609493247771097  | 0.06975313207249972 
     Dxx       |  0.00029670567015240555 | 0.0008561110339464981
     Dxy       |  0.0010397403131681168  | 0.003370609530491976 
     Dyx       |  0.0010396602647615128  | 0.003370170942076215 
     Dyy       |  0.0002922720440920302  | 0.0008425976999418495 
     Exx       |  0.004157529332322421   | 0.012199830469702343 
     Exy       |  0.003932233482201981   | 0.011025843855977937 
     Eyx       |  0.0039323786198353205  | 0.01102631663360245 
     Eyy       |  0.004155897889165329   | 0.012199554698203097 


In [30]:
# Save the loss history in a DataFrame
df = pd.DataFrame({'VMSE_concentration' : VMSE_concentration, 'VMSE_vx' : VMSE_vx, 'VMSE_vy' : VMSE_vy,\
                    'VMSE_Dxx' : VMSE_Dxx, 'VMSE_Dxy' : VMSE_Dxy, \
                    'VMSE_Dyx' : VMSE_Dyx, 'VMSE_Dyy' : VMSE_Dyy, \
                    'VMSE_Exx' : VMSE_Exx, 'VMSE_Exy' : VMSE_Exy, \
                    'VMSE_Eyx' : VMSE_Eyx, 'VMSE_Eyy' : VMSE_Eyy, \
                    'VMSE_concentration_phys' : VMSE_concentration_phys, 'VMSE_vx_phys' : VMSE_vx_phys, 'VMSE_vy_phys' : VMSE_vy_phys,\
                    'VMSE_Dxx_phys' : VMSE_Dxx_phys, 'VMSE_Dxy_phys' : VMSE_Dxy_phys, \
                    'VMSE_Dyx_phys' : VMSE_Dyx_phys, 'VMSE_Dyy_phys' : VMSE_Dyy_phys, \
                    'VMSE_Exx_phys' : VMSE_Exx_phys, 'VMSE_Exy_phys' : VMSE_Exy_phys, \
                    'VMSE_Eyx_phys' : VMSE_Eyx_phys, 'VMSE_Eyy_phys' : VMSE_Eyy_phys })
print(df)
df.to_csv("9_VMSE_UNetConvNext_physics_evaluation.csv", sep=',', float_format='{:.2e}'.format, index=False)

      VMSE_concentration   VMSE_vx   VMSE_vy  VMSE_Dxx  VMSE_Dxy  VMSE_Dyx  \
0           1.886323e-09  0.001494  0.002392  0.000504  0.001848  0.001846   
1           4.806593e-10  0.000604  0.000713  0.000135  0.000399  0.000398   
2           4.484041e-10  0.000627  0.000860  0.000059  0.000341  0.000341   
3           2.776959e-09  0.003237  0.003111  0.000462  0.001051  0.001051   
4           2.324274e-10  0.000390  0.000489  0.000176  0.000439  0.000439   
...                  ...       ...       ...       ...       ...       ...   
996         1.752282e-10  0.000469  0.000433  0.000107  0.000246  0.000246   
997         1.357907e-09  0.001831  0.003943  0.000783  0.002217  0.002216   
998         3.530979e-09  0.002844  0.002079  0.000935  0.002035  0.002035   
999         9.704986e-09  0.001726  0.001499  0.000306  0.001132  0.001133   
1000        2.162569e-08  0.000726  0.001553  0.000596  0.002258  0.002258   

      VMSE_Dyy  VMSE_Exx  VMSE_Exy  VMSE_Eyx  ...  VMSE_vx_phys

## About how preprocess works

The function works in ```torch.Size([4, 4, 256, 256, 11])``` although mu and sigma are ```torch.Size([11])``` due to the logic of torch module. Pytorch align dimensions from **right to left**.

Continuation an example

In [21]:
import torch

# Tensor de tamaño (4, 11, 256, 256)
a = torch.randn(4, 4, 256, 256, 11)

# Tensor de tamaño (11,)
b = torch.arange(11, dtype=torch.float32)

print(a.shape)
print(b.shape)

torch.Size([4, 4, 256, 256, 11])
torch.Size([11])


In [22]:
c = a-b
print(c.shape)

torch.Size([4, 4, 256, 256, 11])


## This postprocess stage is not included in the training

## New objective: Return to the training and include postprocess stage in a proper way. Increase the number of epochs for training.